In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **`INDEX`**

### **1. Importing all the Necessary Libraries**

### **2. Loading the Data**

### **3. Exploratory Data Analysis (EDA)**

### **4. Handling Null Values**

### **5. Train Test Split**

### **6. Data Preprocessing**

### **7. Model Building and Hyperparameter Tuning**

### 1. Importing all the Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

### 2. Loading the Dataset

Here, the training and test datasets are loaded using `pd.read_csv()`. This function reads CSV files into DataFrames.

- `train.csv` contains the training data.
- `test.csv` contains the test data.

Displaying the first few rows of `train` using `.head()` helps verify the data’s structure and content.


In [ ]:
train = pd.read_csv('/kaggle/input/System-Threat-Forecaster/train.csv')
test = pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')

### 3. Exploratory Data Analysis (EDA)

#### Analysing the shape of both the Training and Testing Dataset 

In [ ]:
print(f"The Number of Rows in the Training Dataset are {train.shape[0]} \nand the Number of Columns being - {train.shape[1]}")

In [ ]:
print(f"The Number of Rows in the Test Dataset are {test.shape[0]} \nand the Number of Columns being - {test.shape[1]}")

### `Insights`

##### The Trainig data has 100000 Rows and 76 Columns
##### And the Test dataset has 10000 Rows and 75 Columns

In [ ]:
train.head(n = 3)

### Checking for Null Values

This code cell checks for the percentage of Null values in each column of the `train` dataset using `.isnull().sum()`.

- Null values can affect model performance and may require imputation or handling.


In [ ]:
null_values = (train.isnull().sum()/len(train)*100).sort_values(ascending=False)
print(f"The Percentage of Null Values in Training Data: \n{null_values[null_values > 0]}")

#### **`Insights`**

##### As we can see, the percentage of Null Values in the variables is less than 1 percent

### Statistical Summary of Numerical Columns

The `.describe()` method provides a statistical summary of all numerical columns in the training dataset, including:

- Count, mean, standard deviation, minimum, and maximum values.
- Percentiles (25%, 50%, 75%) which provide insights into the data distribution.


In [ ]:
train.describe()

### Summary of the Dataset

The following code provides an overview of the training dataset using `.info()`, which displays:

- The number of non-null values in each column.
- Data types of the columns.
- Memory usage of the dataset.




In [ ]:
train.info()

### Univariate Analysis

#### Starting with Univariate Analysis, let's have a look at the `target` variable 

#### 1. **`target`**

In [ ]:
train["target"].value_counts(normalize = True)*100

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=train['target'])
plt.title('Distribution of Target')
plt.xlabel('Malware Detected (target)')
plt.ylabel('Count')
plt.show()

### **`Insights`** on the Distribution of the `target` variable

##### The dataset is a balanced one as the percentage 0's and 1's are approximately equal

#### Let us have a look at some of the Categorical Columns

In [ ]:
# Analyzing some categorical columns
cate_features = ['ProductName', 'IsBetaUser', 'RealTimeProtectionState']

# Display value counts for each categorical feature
for col in cate_features:
    print(f"\nValue Counts for: {col}")
    print(train[col].value_counts())
    print(f"The Number of Unique Values in {col} are {train[col].nunique()}")


#### **`Observations`**

**ProductName:**
- Only two unique values exist, with "win8defender" dominating (~99% of records) and "mse" being extremely rare.
- The heavy imbalance suggests that ProductName may have limited predictive power unless the rare category ("mse") shows distinct behavior.

**IsBetaUser:**
- The column contains a single unique value (0) across all records, indicating no beta users in the dataset.
- With no variation present, this feature is unlikely to contribute useful information for modeling and can be dropped.

**RealTimeProtectionState:**
- There are six unique states, but over 97% of the data is concentrated in state 7, indicating a standard configuration.
- The remaining states are very infrequent, which could require special handling or grouping if the feature is to be used in modeling.

#### Analysing some of the Numerical columns

In [ ]:

num_features = ['TotalPhysicalRAMMB', 'PrimaryDiskCapacityMB', 'NumAntivirusProductsInstalled']


for col in num_features:
    print(f"\nValue Counts for: {col}")
    print(train[col].value_counts())
    print(f"The Number of Unique Values in {col} are {train[col].nunique()}")


#### **`Insights`**

**TotalPhysicalRAMMB:**  
- There are 127 unique values, with many systems clustering around common memory sizes such as 4096, 8192, and 2048 MB, indicating standard hardware configurations for many machines.  
- The diversity in values hints at a wide range of system configurations, which could influence performance and may be useful for further segmentation or analysis.

**PrimaryDiskCapacityMB:**  
- With 398 unique values, this feature shows a broad range of disk capacities; a few values like 476940.0 and 953869.0 occur very frequently, representing common disk sizes.  
- The long tail of unique, rarely occurring capacities suggests that while many systems follow standard configurations, there are several outlier setups that might merit additional investigation.

**NumAntivirusProductsInstalled:**  
- The data is concentrated in just 5 unique values, with most systems having only 1 antivirus product installed and a significant portion having 2 products.  
- The very low counts for 4 and 5 antivirus products indicate that having more than 3 products is rare, which might reflect standard consumer practices or system configurations.

In [ ]:
# Calculate skewness and kurtosis for numeric features
skewness = train[num_features].skew()
kurtosis = train[num_features].kurt()

print(f"Skewness of Numeric Features: \n\n{skewness}")
print(f"\nKurtosis of Numeric Features: \n\n{kurtosis}")


#### **From this table we can `observe`:**

**TotalPhysicalRAMMB:**  
1. The skewness is very high at 7.919121, indicating a strong right skew. This suggests that most systems have lower RAM values, but a few systems possess exceptionally high RAM, creating a long right tail.  
2. The kurtosis is extremely high at 229.297303, highlighting a heavy-tailed distribution with numerous outliers. This extreme peakedness suggests that the feature contains many extreme values compared to a normal distribution.

**PrimaryDiskCapacityMB:**  
1. The skewness of 1.219886 indicates a moderate right skew, meaning most disk capacities tend to be lower, with a tail extending towards higher values.  
2. The kurtosis of 5.585474, although not as extreme as for RAM, still suggests a more peaked distribution with heavier tails than a normal distribution, implying the presence of outliers.

**NumAntivirusProductsInstalled:**  
1. The skewness of 1.320539 points to a right-skewed distribution, indicating that the majority of systems have a low count of installed antivirus products, with a few cases having more installed.  
2. The kurtosis of 1.052517 is modestly above that of a normal distribution, suggesting a slight heaviness in the tails. This implies that while most systems follow a common pattern, there are occasional deviations that may need consideration.

**Skewness:**

**0:** Symmetric distribution.

**Positive:** Right-skewed (longer tail on the right).

**Negative:** Left-skewed (longer tail on the left).



**Kurtosis:**

**Near 0** (Excess Kurtosis): Distribution similar to normal.

**Positive:** More peaked with heavy tails (leptokurtic).

**Negative:** Flatter distribution with lighter tails (platykurtic).

#### Bivariate Analysis

In [ ]:
# Crosstab analysis of ProductName vs. target
print(f"Crosstab: ProductName vs Target \n{pd.crosstab(train['ProductName'], train['target'])}")


#### **From this Contingency Table we can see:**

1. The crosstab shows that "win8defender" dominates the dataset with nearly 100,000 records, split almost evenly between non-infected (target 0) and infected (target 1) systems.  
2. In contrast, "mse" has very few records (around 229 in total), suggesting that its influence on the overall model may be minimal due to its low representation.

#### Multivariate Analysis

In [ ]:
# Select a set of features for multivariate analysis
subset_cols = ['TotalPhysicalRAMMB', 'PrimaryDiskCapacityMB', 'NumAntivirusProductsInstalled', 'target']

print("\nCorrelation Matrix for \nTotalPhysicalRAMMB, \nPrimaryDiskCapacityMB, \nNumAntivirusProductsInstalled, \ntarget:")
train[subset_cols].corr()
sns.pairplot(train[subset_cols], hue='target', diag_kind='kde', corner=True)
plt.suptitle('Pairplot for Selected Features', y=1.02)
plt.show()


### 📊 Correlation Heatmap for some Numeric Features

In [ ]:
selected_cols = [
    'ProcessorCoreCount', 'TotalPhysicalRAMMB', 'PrimaryDiskCapacityMB', 'SystemVolumeCapacityMB',
    'PrimaryDisplayDiagonalInches', 'PrimaryDisplayResolutionHorizontal', 'PrimaryDisplayResolutionVertical',
    'OSBuildNumber', 'OSBuildRevisionOnly', 'OSInstallLanguageID', 'RealTimeProtectionState',
    'FirewallEnabled', 'IsSecureBootEnabled', 'IsVirtualDevice', 'IsTouchEnabled',
    'IsPenCapable', 'IsAlwaysOnAlwaysConnectedCapable'
]
# Filtering dataset with selected columns
corr_data = train[selected_cols]

# Compute correlation matrix
corr_matrix = corr_data.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap of Selected Features", fontsize=15)
plt.show()

#### Outliers Detection and Analysis

In [ ]:
# Check for outliers using the IQR method for selected numeric features
for col in num_features:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = train[(train[col] < (Q1 - 1.5 * IQR)) | (train[col] > (Q3 + 1.5 * IQR))]
    print(f"\nOutlier Analysis for {col}:")
    print(f"Number of outliers: {outliers.shape[0]}")
    print(outliers[[col]].describe())

In [ ]:
for col in num_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=train[col])
    plt.title(f'Boxplot for {col} (Outlier Analysis)')
    plt.xlabel(col)
    plt.show()

In [ ]:
# Feature Distributions (Numerical)
numeric_features = train.select_dtypes(include=['int64', 'float64']).columns
numeric_features

In [ ]:
#  Categorical Feature Distributions
categorical_features = train.select_dtypes(include=['object']).columns
categorical_features

In [ ]:
# Convert Date Features to Numeric
def process_dates(df):
    for col in ['DateAS', 'DateOS']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = (df[col] - pd.Timestamp("1970-01-01")).dt.days
    return df

train = process_dates(train)
test = process_dates(test)

### 4. Handling Null Values

In [ ]:
y = train['target']
X = train.drop(columns = ['target'])

In [ ]:
columns_X = X.columns
columns_test_data = test.columns

In [ ]:
si = SimpleImputer(strategy = 'most_frequent')

X = si.fit_transform(X)

test = si.transform(test)

X = pd.DataFrame(X, columns = columns_X)

test = pd.DataFrame(test, columns = columns_test_data)

In [ ]:
test.shape

### 5. Train Test Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 6. Data Preprocessing

In [ ]:
# Define Preprocessing Pipeline
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

#Numerical Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Fill missing values
    ('scaler', StandardScaler())  # Scale data
])

#Categorical Pipeline (One-Hot Encoding instead of LabelEncoder)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Fill missing values
    ('encoder', OneHotEncoder(handle_unknown='ignore'))  # One-Hot Encoding
])

#Column Transformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

### Model Building and Hyperparameter Tuning

#### 1. Logistic Regression

In [ ]:
# Logistic Regression Model
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred_log = log_reg_pipeline.predict(X_val)
print("Logistic Regression Accuracy:", accuracy_score(y_val, y_pred_log))

In [ ]:
print("\nClassification Report:\n", classification_report(y_val, y_pred_log))

#### 2. Random Forest Classifier

In [ ]:
# Random Forest Model
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_val)

print("Random Forest Accuracy:", accuracy_score(y_val, y_pred_rf)) 

In [ ]:
print("\nClassification Report:\n", classification_report(y_val, y_pred_rf))

#### 4. XGBoost Classifier

In [ ]:
# XGBoost Model
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_val)

print("XGBoost Accuracy:", accuracy_score(y_val, y_pred_xgb))

In [ ]:
print("\nClassification Report:\n", classification_report(y_val, y_pred_xgb))

## Hyperparameter Tuning

In [ ]:
# Hyperparameter Tuning for XGBoost
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}

grid_search = GridearchCV(xgb_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters for XGBoost:", grid_search.best_params_)

In [ ]:
#Best Model Prediction
best_xgb_model = grid_search.best_estimator_
y_best_xgb = best_xgb_model.predict(X_val)

print("Best XGBoost Accuracy:", accuracy_score(y_val, y_best_xgb))
print("\nBest Classification Report:\n", classification_report(y_val, y_best_xgb))

# Submission file

In [ ]:
# Initialize and fit the DummyClassifier
# dummy_model = DummyClassifier(strategy="most_frequent")  # "most_frequent" predicts the most common class
# dummy_model.fit(X_train, y_train)

# # Predict on the test data
y_pred = dummy_model.predict(X_test)
Predict on test data
y_pred_test = best_xgb_model.predict(test.drop(columns=['MachineID'], errors='ignore'))


# Create submission DataFrame
submission = pd.DataFrame({'id': range(10000), 'target': y_pred_test})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file created successfully!")